In [ ]:
# [목적] 고정 예시를 직접 넣는 Few-shot Prompt와 LCEL 체인 실행 방법을 실습합니다.
# Few-shot은 모델에 질문·답변 예시를 먼저 보여 주어 원하는 추론 방식과 출력 형태를 따라 하게 만드는 방법입니다.
# 이 셀에서 예시 데이터와 표시 형식을 준비하며, 결과는 다음 셀의 최종 프롬프트 구성에 사용합니다.

from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote import logging
from dotenv import load_dotenv

# LangSmith에 실행 기록을 남겨 프롬프트와 응답을 나중에 확인할 수 있게 합니다.
logging.langsmith("Chapter5-Prompt")

# examples는 모델이 참고할 고정 예시 목록이며, 각 항목은 question과 answer를 한 쌍으로 가집니다.
examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
                추가 질문: 스티브 잡스는 몇 살에 사망했나요?
                중간 답변: 스티브 잡스는 56세에 사망했습니다.
                추가 질문: 아인슈타인은 몇 살에 사망했나요?
                중간 답변: 아인슈타인은 76세에 사망했습니다.
                최종 답변은: 아인슈타인
                """,
    },
]

# {question}, {answer} 자리에 각 예시의 실제 내용을 넣는 틀을 만듭니다.
examples_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

# **는 딕셔너리의 question과 answer를 같은 이름의 자리표시자에 나누어 넣습니다.
print(examples_prompt.format(**examples[0]))

In [ ]:
# [목적] 고정 예시와 새 질문을 합쳐 모델에 전달할 Few-shot Prompt를 완성합니다.
# FewShotPromptTemplate이 예시를 정해진 형식으로 배치하고 suffix에 실제 질문을 붙여 하나의 입력 문자열을 만듭니다.
# 완성된 final_prompt를 먼저 출력해 확인한 뒤, 다음 셀에서 ChatOpenAI의 입력으로 사용합니다.

prompt = FewShotPromptTemplate(
    # examples의 모든 고정 예시를 examples_prompt 형식으로 변환합니다.
    examples=examples,
    example_prompt=examples_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
# format은 {question} 자리를 실제 질문으로 바꿔 하나의 문자열을 완성합니다.
final_prompt = prompt.format(question=question)
print(final_prompt)

In [ ]:
# [목적] 완성된 Few-shot Prompt를 ChatOpenAI에 보내고 생성되는 답변을 실시간으로 확인합니다.
# llm은 프롬프트를 해석해 답변을 만드는 모델 객체이고, stream은 결과를 완성 전부터 작은 조각으로 돌려줍니다.
# 반환된 answer 스트림은 stream_response가 화면에 이어 붙여 사용자에게 바로 보여 줍니다.

from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response

# 별도 모델명을 지정하지 않으면 라이브러리의 기본 채팅 모델 설정을 사용합니다.
llm = ChatOpenAI()

# 앞 셀에서 만든 문자열 전체가 모델 입력으로 전달됩니다.
answer = llm.stream(final_prompt)
stream_response(answer)

In [ ]:
# [목적] Few-shot Prompt 생성, 모델 호출, 문자열 변환을 LCEL 체인 하나로 연결합니다.
# LCEL의 | 연산자는 앞 단계 결과를 다음 단계 입력으로 넘겨 여러 작업을 정해진 순서대로 자동 실행합니다.
# 완성된 chain에는 질문만 전달하면 전체 흐름이 실행되며, 최종 문자열 답변을 스트리밍으로 확인할 수 있습니다.

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=examples_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

# 모델 응답 객체에서 실제 답변 문장만 꺼내도록 StrOutputParser를 마지막에 연결합니다.
chain = prompt | llm | StrOutputParser()

answer = chain.stream(
    # 딕셔너리의 question 값이 프롬프트 안의 {question} 자리에 들어갑니다.
    {"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)
stream_response(answer)